#Preliminary Analysis of Habitat Modification Scores 
####(main dataset is rss_tbl_river_habitat_survey - hms_class and subscores)%md


River Habitat Survey HMS c-  channel resectioning and artificial bank reinforcement

In [0]:
%r
# Load packages
library(sparklyr) # connect to databricks
library(data.table) # data management
library(ggplot2)
# GAMS - use mgcv for ordinal/catagorial data
#library(mgcv)
# Create a sparklyr connection.
sc <- spark_connect(method = "databricks")

In [0]:
# set parameters
param <- list(st="RSN")

# River habitat survey
# hms_class
rss_dt <- sdf_sql(sc, paste0("SELECT *
FROM prd_dash_lab.seda_restricted.rss_tbl_river_habitat_survey
WHERE network_id LIKE '", param$st, 
"%' AND completeness_band NOT LIKE '%Unacceptable%' ")) |> as.data.table()

In [0]:
# change to factors
rss_dt[, network_id := as.factor(network_id)]

In [0]:
#return only hms_class and hms_class_desc columns
rss_dt[, .(hms_class, hms_class_desc), keyby=hms_class] |> unique()

In [0]:
hms_class_table <- table(rss_dt[, hms_class])
# as %
hms_class_table  <- hms_class_table  / sum(hms_class_table) * 100
# Add names
classes <- c("Pristine/Near-natural", "Predominantly unmodified", "Obviously modified", "Significantly modified", "Severely modified")
names(hms_class_table) <- classes
# View table
hms_class_table

In [0]:
ggplot(rss_dt, aes(x = hms_rsctned_bnk_subscore,y = factor(hms_class),colour = factor(hms_class))) +
  geom_jitter(width = 0, height = 0.15,alpha = 0.5,size = 2) +
  scale_colour_manual(values = cols) +
  scale_y_discrete(
    labels = c(
      "1" = "Pristine",
      "2" = "Predominantly unmodified",
      "3" = "Obviously modified",
      "4" = "Significantly modified",
      "5" = "Severely modified" )) +
  labs(
    x = "Channel resectioning sub-score",
    y = "HMS class",
    colour = "HMS Class") +
  theme_bw()

In [0]:
cols <- c("#2ca25f", "#99d8c9", "#fec44f", "#fe9929", "#d95f0e")

ggplot(rss_dt, aes(x = factor(hms_class), y = hms_rsctned_bnk_subscore, colour = factor(hms_class))) +
  scale_colour_manual(values = cols) +
  geom_jitter(
    width = 0.15,
    alpha = 0.4,
    size = 2) +
  labs(
    x = "HMS Class",
    y = "Channel resectioning sub-score",
    title = "Channel resectioning sub-score by HMS class") +
  theme_bw()


We are not able to use the sunscore value directly to categorise how much/to what extent (in a categorical way) that the rivers are affected as this score feeds into the overall HMS score/class. There is Channel Resectioning Index that can be calculated but don't think we have the right data for this (part of hydromorphological indices). 

In [0]:
# table showing no. of sites/network that have scored 0 for channel resectioning - no modifications at all. >0 there are modifications
resection_table <- rss_dt[
  !is.na(hms_rsctned_bnk_subscore),
  .( `Number of networks` = uniqueN(network_id)),by = .
  (`Resectioning` = ifelse(
      hms_rsctned_bnk_subscore == 0,
      "0",
      ">0"))]

resection_table

In [0]:
#as percentage
resection_table[, Percentage := 
  round(`Number of networks` / sum(`Number of networks`) * 100, 1)]

resection_table

In [0]:
ggplot(resection_table,
       aes(x = Resectioning,
           y = Percentage,
           fill = Resectioning)) +
  geom_col(width = 0.7) +
  geom_text(
    aes(label = paste0(Percentage, "%")),
    vjust = -0.5,
    size = 4) +
  scale_fill_manual(
    values = c("0" = "#2ca25f",
               ">0" = "#d95f0e")) +
  labs(
    x = "Channel resectioning",
    y = "Proportion of rivers (%)",
    title = "Proportion of rivers affected by channel resectioning") +
  theme_bw() +
  theme(
    legend.position = "none")

In [0]:
ggplot(rss_dt,
       aes(x = 1,
           y = hms_rsctned_bnk_subscore)) +
  geom_jitter(
    width = 0.15,
    alpha = 0.4,
    size = 2,
    colour = "#d95f0e") +
  labs(
    x = NULL,
    y = "Channel resectioning sub-score",
    title = "Distribution of channel resectioning sub-scores") +
  theme_bw() +
  theme(
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank())

####Subscores grouped based on ESME report

5 = 0 

4 = 1-200 

3 = 201-600 

2 = 601-1400 

1 > 1400 (Maximum score is 2800). 

https://defra.sharepoint.com/:w:/r/sites/WorkDelivery886/_layouts/15/Doc.aspx?sourcedoc=%7B42DE3979-D773-4906-9A7E-B9D0585D42D0%7D&file=ESME%20Methodology%20report%20Final%20V1.2.docx&action=default&mobileredirect=true&DefaultItemOpen=1 - but this includes bed and bank - the dataset here only says bank, is this the same? Also max score is 2800 so that outlier at 4000 was a wrong input?

In [0]:
#re-doing above plots based on the categories

rss_dt[, resectioning_class := fifelse(
  is.na(hms_rsctned_bnk_subscore), NA_integer_,
  fifelse(hms_rsctned_bnk_subscore == 0, 5L,
  fifelse(hms_rsctned_bnk_subscore <= 200, 4L,
  fifelse(hms_rsctned_bnk_subscore <= 600, 3L,
  fifelse(hms_rsctned_bnk_subscore <= 1400, 2L, 
  1L)))))]

In [0]:
resection_summary <- rss_dt[
  !is.na(resectioning_class),
  .(
    networks = uniqueN(network_id)
  ),
  by = resectioning_class]

resection_summary[, percentage :=
  round(networks / sum(networks) * 100, 1)]

resection_summary[order(resectioning_class)]

In [0]:
ggplot(hms_plot_data, aes(x = hms_class, y = percentage,fill = hms_class)) + geom_col(colour = "black") +
  scale_fill_manual(values = cols) +
  scale_y_continuous(breaks = seq(0, 100, 20),limits = c(0, 100),expand = expansion(mult = c(0, 0.05))) + 
  labs(x = "HMS Class", y = "Proportion of Rivers (%)",fill = "HMS Class") +
  geom_text(aes(label = paste0(round(percentage, 1), "%")),vjust = -0.4,size = 3.5) +
  theme_bw() +
  theme(axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      size = 10),
    legend.position = "none")

In [0]:
#plot bar plot and percentages
resection_plot_data <- rss_dt[
  !is.na(resectioning_class),
  .(n = uniqueN(network_id)),
  by = resectioning_class]

resection_plot_data[, percentage := n / sum(n) * 100]

resection_plot_data[, resectioning_class :=
  factor(
    resectioning_class,
    levels = c(5, 4, 3, 2, 1),
    labels = c(
      "0",
      "1–200",
      "201–600",
      "601–1400",
      ">1400"))]

In [0]:
#reverse the colours
cols_resection <- c( "#2ca25f","#99d8c9","#fec44f","#fe9929", "#d95f0e")

#error bars
res <- rss_dt[!is.na(resectioning_class), .N, by = resectioning_class][order(resectioning_class)]
res[, p := N/sum(N)]
res[, c("lo","hi") := {ci <- sapply(N, \(n) binom.test(n, sum(N))$conf.int); .(ci[1,],ci[2,])}]
res[, resectioning_class := factor(resectioning_class, levels = c(5,4,3,2,1), labels = c("0","1-200","201-600","601-1400",">1400"))]

resection_plot_data <- rss_dt[!is.na(resectioning_class), .(n = uniqueN(network_id)), by = resectioning_class]
resection_plot_data[, percentage := n/sum(n)*100]
resection_plot_data[, resectioning_class := factor(resectioning_class, levels = c(5,4,3,2,1), labels = c("0","1-200","201-600","601-1400",">1400"))]

resection_plot_data <- merge(resection_plot_data, res[, .(resectioning_class,lo,hi)], by = "resectioning_class", all.x = TRUE)
resection_plot_data[, `:=`(lower_ci = lo*100, upper_ci = hi*100)]

ggplot(
  resection_plot_data,
  aes(x = resectioning_class, y = percentage, fill = resectioning_class)) +
  geom_col(colour = "black") +
  scale_fill_manual(values = cols_resection) +
  scale_y_continuous(breaks = seq(0, 100, 20), limits = c(0, 100), expand = expansion(mult = c(0, 0.05))) +
  labs( x = "Bank and bed resectioning sub-score",y = "Proportion of Rivers (%)",fill = "Resectioning class") +
  geom_text(aes(label = paste0(round(percentage, 1), "%")),vjust = -2.5 ,size = 3.5) +
  geom_errorbar(aes(ymin = lower_ci, ymax = upper_ci), width = 0.1, linewidth = 0.25) +
  theme_bw() +
  theme(
    axis.text.x = element_text(
      angle = 45,
      hjust = 1,
      size = 10),
    legend.position = "none")

In [0]:
resection_plot_data$resectioning_class
res$resectioning_class

####Estimated proportion of river length affected by artificial bank reinforcement - looked at hms_reforced_bnk_bed_subscore

In [0]:
ggplot(rss_dt, aes(x = factor(hms_class), y = hms_reforced_bnk_bed_subscore, colour = factor(hms_class))) +
  scale_colour_manual(values = cols) +
  geom_jitter(
    width = 0.15,
    alpha = 0.4,
    size = 2) +
  labs(
    x = "HMS Class",
    y = "Channel reinforced sub-score",
    title = "Channel reinforcement sub-score by HMS class") +
  theme_bw()


In [0]:
# similar to avove, table showing no. of sites/network that have scored 0 for channel resectioning - no modifications at all. >0 there are modifications
reinforce_table <- rss_dt[
  !is.na(hms_reforced_bnk_bed_subscore),
  .( `Number of networks` = uniqueN(network_id)),by = .
  (`Reinforcing` = ifelse(
      hms_reforced_bnk_bed_subscore == 0,
      "0",
      ">0"))]

reinforce_table

In [0]:
#as percentage
reinforce_table[, Percentage := 
  round(`Number of networks` / sum(`Number of networks`) * 100, 1)]

reinforce_table

In [0]:
ggplot(reinforce_table, aes(x = Reinforcing, y = Percentage, fill = Reinforcing)) +
  geom_col(width = 0.7) +
  geom_text(aes(label = paste0(Percentage, "%")), vjust = -0.5, size = 4) +
  scale_fill_manual(values = c("0" = "#2ca25f", ">0" = "#d95f0e")) +
  labs(
    x = "Channel Reinforcing",
    y = "Proportion of rivers (%)",
    title = "Proportion of rivers affected by channel Reinforcing") +
  theme_bw() +
  theme(legend.position = "none")